In [12]:
import time
import requests
from urllib.parse import quote

API_KEY = "a52e01cb767cf5aa497eeef089d3595a"
INST_TOKEN = None  # Elsevier inst token if you have one

BASE_URL = "https://api.elsevier.com/content/article/doi/"

def elsevier_get(url, headers, params=None, retries=3, timeout=30):
    """GET with simple retry/backoff handling for rate limits and transient failures."""
    for attempt in range(retries):
        r = requests.get(url, headers=headers, params=params, timeout=timeout)

        # Rate limit
        if r.status_code == 429:
            wait = int(r.headers.get("Retry-After", 2 ** attempt))
            time.sleep(wait)
            continue

        # transient errors
        if r.status_code in (500, 502, 503, 504):
            time.sleep(2 ** attempt)
            continue

        return r
    return r  # return last response

def get_fulltext_by_doi(doi):
    doi_safe = quote(doi, safe="")
    url = f"{BASE_URL}{doi_safe}"

    headers = {"X-ELS-APIKey": API_KEY}
    if INST_TOKEN:
        headers["X-ELS-Insttoken"] = INST_TOKEN

    # 1) Try plain text
    headers_txt = {**headers, "Accept": "text/plain"}
    r_txt = elsevier_get(url, headers_txt, params={"view": "FULL"})
    if r_txt.ok and r_txt.text.strip():
        return r_txt.text, "text/plain"

    # 2) Try XML
    headers_xml = {**headers, "Accept": "text/xml"}
    r_xml = elsevier_get(url, headers_xml, params={"view": "FULL"})
    if r_xml.ok and r_xml.text.strip():
        return r_xml.text, "text/xml"

    # 3) Try JSON FULL view
    headers_json = {**headers, "Accept": "application/json"}
    r_json = elsevier_get(url, headers_json, params={"view": "FULL"})
    if r_json.ok:
        j = r_json.json()
        resp = j.get("full-text-retrieval-response", {})

        # Try multiple likely keys
        for key in ("originalText", "originalXml"):
            if key in resp and resp[key]:
                return resp[key], f"application/json:{key}"

        # fallback: sometimes content is nested
        core = resp.get("coredata", {})
        if core:
            return str(core), "application/json:coredata"

    # If nothing worked, return status/debug info
    return None, f"Failed. Status codes: txt={r_txt.status_code}, xml={r_xml.status_code}, json={r_json.status_code}"

def save_markdown(doi, filename="paper.md"):
    content, ctype = get_fulltext_by_doi(doi)
    if not content:
        print(f"Could not fetch full text for DOI {doi}.")
        print(ctype)
        return

    with open(filename, "w", encoding="utf-8") as f:
        f.write(f"# Full Text for DOI {doi}\n\n")
        f.write(f"**Content type:** {ctype}\n\n")
        f.write(content)

    print(f"Saved {ctype} to {filename}")

# Example:
save_markdown("10.1016/j.jmrt.2025.11.003", "paper.md")


Saved text/plain to paper.md


In [ ]:
import os, re, json
import requests
from urllib.parse import quote
from bs4 import BeautifulSoup

INST_TOKEN = None
BASE_URL = "https://api.elsevier.com/content/article/doi/"

def safe_id(s):
    """Convert DOI into a filesystem-safe string."""
    return re.sub(r"[^a-zA-Z0-9._-]+", "_", s)

def fetch_xml_by_doi(doi):
    doi_safe = quote(doi, safe="")
    url = f"{BASE_URL}{doi_safe}"

    headers = {"X-ELS-APIKey": API_KEY, "Accept": "text/xml"}
    if INST_TOKEN:
        headers["X-ELS-Insttoken"] = INST_TOKEN

    r = requests.get(url, headers=headers, params={"view": "FULL"}, timeout=30)
    if r.ok and r.text.strip():
        return r.text
    return None

def normalize_text(s):
    """Normalize whitespace."""
    s = re.sub(r"\s+", " ", s)
    return s.strip()

def rows_to_markdown(rows):
    """Convert rows (list[list[str]]) to Markdown table."""
    if not rows:
        return ""

    # find max columns
    ncol = max(len(r) for r in rows)
    norm = [r + [""]*(ncol-len(r)) for r in rows]

    header = norm[0]
    body = norm[1:] if len(norm) > 1 else []

    md = []
    md.append("| " + " | ".join(header) + " |")
    md.append("| " + " | ".join(["---"] * ncol) + " |")
    for r in body:
        md.append("| " + " | ".join(r) + " |")

    return "\n".join(md)

def extract_table_rows(table_tag):
    """
    Extract table rows from multiple possible Elsevier/JATS table formats:
    - ce:table + ce:row + ce:entry
    - table-wrap / table / tr / td / th
    """
    rows = []

    # common row tags
    row_tags = table_tag.find_all(["ce:row", "row", "tr"])
    for row in row_tags:
        cells = []
        cell_tags = row.find_all(["ce:entry", "entry", "td", "th"])
        for c in cell_tags:
            cells.append(normalize_text(c.get_text(" ", strip=True)))
        if any(cells):
            rows.append(cells)

    return rows

def extract_caption(table_tag):
    cap = table_tag.find(["ce:caption", "caption", "title"])
    if cap:
        return normalize_text(cap.get_text(" ", strip=True))
    return None

def extract_tables_from_xml(soup):
    """Extract tables and return list of {table_index, caption, rows}."""
    tables = soup.find_all(["ce:table", "table-wrap", "table"])

    extracted = []
    idx = 1
    for t in tables:
        rows = extract_table_rows(t)
        if not rows:
            continue

        caption = extract_caption(t)
        extracted.append({
            "table_index": idx,
            "caption": caption,
            "rows": rows
        })
        idx += 1

    return extracted

def extract_sections_as_markdown(soup, tables_map):
    """
    Convert XML content into Markdown with headings + paragraphs,
    and replace tables with placeholder links (or inline markdown).
    """

    md_lines = []

    # Article title (many formats)
    title_tag = soup.find(["dc:title", "title", "ce:title", "article-title"])
    if title_tag:
        md_lines.append("# " + normalize_text(title_tag.get_text(" ", strip=True)))
        md_lines.append("")

    # Elsevier sections often use <ce:section> with <ce:section-title>
    sections = soup.find_all(["ce:section", "sec"])

    if not sections:
        # fallback: dump all paragraphs
        paras = soup.find_all(["ce:para", "p"])
        for p in paras:
            txt = normalize_text(p.get_text(" ", strip=True))
            if txt:
                md_lines.append(txt)
                md_lines.append("")
        return "\n".join(md_lines)

    for sec in sections:
        # section title
        st = sec.find(["ce:section-title", "title"])
        if st:
            md_lines.append("## " + normalize_text(st.get_text(" ", strip=True)))
            md_lines.append("")

        # paragraph content inside this section
        for p in sec.find_all(["ce:para", "p"], recursive=True):
            txt = normalize_text(p.get_text(" ", strip=True))
            if txt:
                md_lines.append(txt)
                md_lines.append("")

        # tables within section
        for table in sec.find_all(["ce:table", "table-wrap", "table"], recursive=True):
            # match this table object to extracted table index by content
            # simplest: generate a key by caption+first row
            cap = extract_caption(table)
            rows = extract_table_rows(table)
            if not rows:
                continue

            key = (cap or "") + "|" + "|".join(rows[0])
            if key in tables_map:
                tinfo = tables_map[key]
                idx = tinfo["table_index"]
                md_lines.append(f"### Table {idx}")
                if tinfo.get("caption"):
                    md_lines.append(f"**Caption:** {tinfo['caption']}")
                md_lines.append("")
                md_lines.append(rows_to_markdown(tinfo["rows"]))
                md_lines.append("")
    return "\n".join(md_lines)

def save_paper_as_markdown_and_tables(doi, outdir="output"):
    paper_id = safe_id(doi)
    base_dir = os.path.join(outdir, paper_id)
    table_dir = os.path.join(base_dir, "tables")
    os.makedirs(table_dir, exist_ok=True)

    xml_text = fetch_xml_by_doi(doi)
    if not xml_text:
        print("Failed to fetch XML full-text.")
        return

    # save raw XML
    xml_path = os.path.join(base_dir, "paper.xml")
    with open(xml_path, "w", encoding="utf-8") as f:
        f.write(xml_text)

    soup = BeautifulSoup(xml_text, "xml")

    # --------------------------------------------------
    # Normalize in-text citations BEFORE extraction
    # --------------------------------------------------
    ref_map = build_refid_to_label_map(soup)
    normalize_cross_refs_in_soup(soup, ref_map)

    # Extract tables first
    tables = extract_tables_from_xml(soup)

    # Create key map to match embedded tables
    tables_map = {}
    for t in tables:
        cap = t.get("caption") or ""
        key = cap + "|" + "|".join(t["rows"][0])
        tables_map[key] = t

    # Save each table separately
    for t in tables:
        idx = t["table_index"]
        json_path = os.path.join(table_dir, f"table_{idx:03d}.json")
        md_path = os.path.join(table_dir, f"table_{idx:03d}.md")

        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(t, f, ensure_ascii=False, indent=2)

        md_content = ""
        if t.get("caption"):
            md_content += f"**Caption:** {t['caption']}\n\n"
        md_content += rows_to_markdown(t["rows"])

        with open(md_path, "w", encoding="utf-8") as f:
            f.write(md_content)

    # Convert main XML to Markdown (with tables inline)
    md_text = extract_sections_as_markdown(soup, tables_map)

    md_file = os.path.join(base_dir, "paper.md")
    with open(md_file, "w", encoding="utf-8") as f:
        f.write(md_text)

    print(f"Saved paper.md and {len(tables)} tables to: {base_dir}")

# Example usage
save_paper_as_markdown_and_tables("10.1016/j.jmrt.2025.11.003")


Saved paper.md and 3 tables to: output/10.1016_j.jmrt.2025.11.003


In [15]:
import os, re, json
import requests
from urllib.parse import quote
from bs4 import BeautifulSoup

API_KEY = "a52e01cb767cf5aa497eeef089d3595a"
INST_TOKEN = None
BASE_URL = "https://api.elsevier.com/content/article/doi/"

# ----------------------------
# Helpers
# ----------------------------
def safe_id(s):
    """Convert DOI into a filesystem-safe string."""
    return re.sub(r"[^a-zA-Z0-9._-]+", "_", s)

def safe_filename(title):
    """Convert section title into a filesystem-safe filename."""
    if not title:
        return "Untitled"
    title = re.sub(r"\s+", " ", title).strip()
    title = re.sub(r"[^\w\-_ .]", "_", title)
    return title[:80].strip()

def fetch_xml_by_doi(doi):
    doi_safe = quote(doi, safe="")
    url = f"{BASE_URL}{doi_safe}"

    headers = {"X-ELS-APIKey": API_KEY, "Accept": "text/xml"}
    if INST_TOKEN:
        headers["X-ELS-Insttoken"] = INST_TOKEN

    r = requests.get(url, headers=headers, params={"view": "FULL"}, timeout=30)
    if r.ok and r.text.strip():
        return r.text
    return None

def normalize_text(s):
    """Normalize whitespace."""
    s = re.sub(r"\s+", " ", s)
    return s.strip()

def rows_to_markdown(rows):
    """Convert rows (list[list[str]]) to Markdown table."""
    if not rows:
        return ""

    ncol = max(len(r) for r in rows)
    norm = [r + [""] * (ncol - len(r)) for r in rows]

    header = norm[0]
    body = norm[1:] if len(norm) > 1 else []

    md = []
    md.append("| " + " | ".join(header) + " |")
    md.append("| " + " | ".join(["---"] * ncol) + " |")
    for r in body:
        md.append("| " + " | ".join(r) + " |")

    return "\n".join(md)

# ----------------------------
# Table extraction
# ----------------------------
def extract_table_rows(table_tag):
    rows = []
    row_tags = table_tag.find_all(["ce:row", "row", "tr"])
    for row in row_tags:
        cells = []
        cell_tags = row.find_all(["ce:entry", "entry", "td", "th"])
        for c in cell_tags:
            cells.append(normalize_text(c.get_text(" ", strip=True)))
        if any(cells):
            rows.append(cells)
    return rows

def extract_caption(table_tag):
    cap = table_tag.find(["ce:caption", "caption", "title"])
    if cap:
        return normalize_text(cap.get_text(" ", strip=True))
    return None

def extract_tables_from_xml(soup):
    tables = soup.find_all(["ce:table", "table-wrap", "table"])
    extracted = []
    idx = 1

    for t in tables:
        rows = extract_table_rows(t)
        if not rows:
            continue
        caption = extract_caption(t)

        extracted.append({
            "table_index": idx,
            "caption": caption,
            "rows": rows
        })
        idx += 1

    return extracted

# ----------------------------
# Abstract extraction
# ----------------------------
def extract_abstract_from_xml(soup):
    abs_tag = soup.find(["ce:abstract", "abstract"])
    if abs_tag:
        paras = abs_tag.find_all(["ce:para", "p"])
        if paras:
            return "\n\n".join(normalize_text(p.get_text(" ", strip=True)) for p in paras if p.get_text(strip=True))
        return normalize_text(abs_tag.get_text(" ", strip=True))

    dc_desc = soup.find("dc:description")
    if dc_desc and dc_desc.get_text(strip=True):
        return normalize_text(dc_desc.get_text(" ", strip=True))

    return None

# ----------------------------
# Section extraction
# ----------------------------
def section_to_markdown(sec, tables_map):
    """
    Convert a <ce:section> or <sec> block into Markdown text.
    Tables inside sections are embedded inline.
    """
    md_lines = []

    # Section title
    st = sec.find(["ce:section-title", "title"])
    if st:
        md_lines.append("## " + normalize_text(st.get_text(" ", strip=True)))
        md_lines.append("")

    # Paragraphs
    for p in sec.find_all(["ce:para", "p"], recursive=True):
        txt = normalize_text(p.get_text(" ", strip=True))
        if txt:
            md_lines.append(txt)
            md_lines.append("")

    # Tables
    for table in sec.find_all(["ce:table", "table-wrap", "table"], recursive=True):
        cap = extract_caption(table)
        rows = extract_table_rows(table)
        if not rows:
            continue

        key = (cap or "") + "|" + "|".join(rows[0])
        if key in tables_map:
            tinfo = tables_map[key]
            idx = tinfo["table_index"]

            md_lines.append(f"### Table {idx}")
            if tinfo.get("caption"):
                md_lines.append(f"**Caption:** {tinfo['caption']}")
            md_lines.append("")
            md_lines.append(rows_to_markdown(tinfo["rows"]))
            md_lines.append("")

    return "\n".join(md_lines).strip()

# ----------------------------
# Main runner
# ----------------------------
def save_paper_as_markdown_and_tables(doi, outdir="output"):
    paper_id = safe_id(doi)
    base_dir = os.path.join(outdir, paper_id)

    sections_dir = os.path.join(base_dir, "sections")
    table_dir = os.path.join(base_dir, "tables")

    os.makedirs(sections_dir, exist_ok=True)
    os.makedirs(table_dir, exist_ok=True)

    xml_text = fetch_xml_by_doi(doi)
    if not xml_text:
        print("Failed to fetch XML full-text.")
        return

    # Save raw XML
    xml_path = os.path.join(base_dir, "paper.xml")
    with open(xml_path, "w", encoding="utf-8") as f:
        f.write(xml_text)

    soup = BeautifulSoup(xml_text, "xml")

    # Article title
    title_tag = soup.find(["dc:title", "title", "ce:title", "article-title"])
    paper_title = normalize_text(title_tag.get_text(" ", strip=True)) if title_tag else "Untitled Paper"

    # Extract tables first
    tables = extract_tables_from_xml(soup)

    # Table map for embedding
    tables_map = {}
    for t in tables:
        cap = t.get("caption") or ""
        key = cap + "|" + "|".join(t["rows"][0])
        tables_map[key] = t

    # Save each table separately
    for t in tables:
        idx = t["table_index"]
        json_path = os.path.join(table_dir, f"table_{idx:03d}.json")
        md_path = os.path.join(table_dir, f"table_{idx:03d}.md")

        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(t, f, ensure_ascii=False, indent=2)

        md_content = ""
        if t.get("caption"):
            md_content += f"**Caption:** {t['caption']}\n\n"
        md_content += rows_to_markdown(t["rows"])

        with open(md_path, "w", encoding="utf-8") as f:
            f.write(md_content)

    # ----------------------------
    # Save Abstract as Section file
    # ----------------------------
    abstract_text = extract_abstract_from_xml(soup)
    combined_sections_md = [f"# {paper_title}", ""]

    if abstract_text:
        abs_md = "# Abstract\n\n" + abstract_text.strip()

        abs_path = os.path.join(sections_dir, "000_Abstract.md")
        with open(abs_path, "w", encoding="utf-8") as f:
            f.write(abs_md)

        combined_sections_md.append(abs_md)
        combined_sections_md.append("")

    # ----------------------------
    # Extract and save all sections separately
    # ----------------------------
    sections = soup.find_all(["ce:section", "sec"])
    sec_counter = 1

    for sec in sections:
        sec_title_tag = sec.find(["ce:section-title", "title"])
        sec_title = normalize_text(sec_title_tag.get_text(" ", strip=True)) if sec_title_tag else f"Section_{sec_counter}"

        md_text = section_to_markdown(sec, tables_map)
        if not md_text.strip():
            continue

        fname = f"{sec_counter:03d}_{safe_filename(sec_title)}.md"
        sec_path = os.path.join(sections_dir, fname)

        with open(sec_path, "w", encoding="utf-8") as f:
            f.write(md_text)

        combined_sections_md.append(md_text)
        combined_sections_md.append("")
        sec_counter += 1

    # Save combined paper.md
    md_file = os.path.join(base_dir, "paper.md")
    with open(md_file, "w", encoding="utf-8") as f:
        f.write("\n".join(combined_sections_md).strip())

    print(f"✅ Saved:\n- paper.xml\n- paper.md\n- {sec_counter-1} section files in sections/\n- {len(tables)} tables in tables/\n➡️ Output: {base_dir}")

# Example usage
save_paper_as_markdown_and_tables("10.1016/j.jmrt.2025.11.003")


✅ Saved:
- paper.xml
- paper.md
- 11 section files in sections/
- 3 tables in tables/
➡️ Output: output/10.1016_j.jmrt.2025.11.003


In [16]:
import os, re, json
import requests
from urllib.parse import quote
from bs4 import BeautifulSoup

API_KEY = "a52e01cb767cf5aa497eeef089d3595a"
INST_TOKEN = None
BASE_URL = "https://api.elsevier.com/content/article/doi/"

# -------------------------------------------------
# Helpers
# -------------------------------------------------
def safe_id(s):
    """Convert DOI into a filesystem-safe string."""
    return re.sub(r"[^a-zA-Z0-9._-]+", "_", s)

def safe_filename(title):
    """Convert section title into filesystem-safe filename."""
    if not title:
        return "Untitled"
    title = re.sub(r"\s+", " ", title).strip()
    title = re.sub(r"[^\w\-_ .]", "_", title)
    return title[:80].strip()

def normalize_text(s):
    """Normalize whitespace."""
    s = re.sub(r"\s+", " ", s)
    return s.strip()

def fetch_xml_by_doi(doi):
    doi_safe = quote(doi, safe="")
    url = f"{BASE_URL}{doi_safe}"

    headers = {"X-ELS-APIKey": API_KEY, "Accept": "text/xml"}
    if INST_TOKEN:
        headers["X-ELS-Insttoken"] = INST_TOKEN

    r = requests.get(url, headers=headers, params={"view": "FULL"}, timeout=30)
    if r.ok and r.text.strip():
        return r.text
    return None

# -------------------------------------------------
# Table Extraction
# -------------------------------------------------
def rows_to_markdown(rows):
    if not rows:
        return ""
    ncol = max(len(r) for r in rows)
    norm = [r + [""] * (ncol - len(r)) for r in rows]

    header = norm[0]
    body = norm[1:] if len(norm) > 1 else []

    md = []
    md.append("| " + " | ".join(header) + " |")
    md.append("| " + " | ".join(["---"] * ncol) + " |")
    for r in body:
        md.append("| " + " | ".join(r) + " |")
    return "\n".join(md)

def extract_table_rows(table_tag):
    rows = []
    row_tags = table_tag.find_all(["ce:row", "row", "tr"])
    for row in row_tags:
        cells = []
        cell_tags = row.find_all(["ce:entry", "entry", "td", "th"])
        for c in cell_tags:
            cells.append(normalize_text(c.get_text(" ", strip=True)))
        if any(cells):
            rows.append(cells)
    return rows

def extract_caption(table_tag):
    cap = table_tag.find(["ce:caption", "caption", "title"])
    if cap:
        return normalize_text(cap.get_text(" ", strip=True))
    return None

def extract_tables_from_xml(soup):
    tables = soup.find_all(["ce:table", "table-wrap", "table"])
    extracted = []
    idx = 1
    for t in tables:
        rows = extract_table_rows(t)
        if not rows:
            continue
        caption = extract_caption(t)
        extracted.append({
            "table_index": idx,
            "caption": caption,
            "rows": rows
        })
        idx += 1
    return extracted

# -------------------------------------------------
# Abstract Extraction
# -------------------------------------------------
def extract_abstract_from_xml(soup):
    abs_tag = soup.find(["ce:abstract", "abstract"])
    if abs_tag:
        paras = abs_tag.find_all(["ce:para", "p"])
        if paras:
            return "\n\n".join(normalize_text(p.get_text(" ", strip=True)) for p in paras if p.get_text(strip=True))
        return normalize_text(abs_tag.get_text(" ", strip=True))

    dc_desc = soup.find("dc:description")
    if dc_desc and dc_desc.get_text(strip=True):
        return normalize_text(dc_desc.get_text(" ", strip=True))

    return None

# -------------------------------------------------
# Section + Subsection Extraction (Recursive)
# -------------------------------------------------
SECTION_TAGS = ["ce:section", "sec"]
TITLE_TAGS = ["ce:section-title", "title"]
PARA_TAGS = ["ce:para", "p"]

def get_section_title(sec):
    title_tag = sec.find(TITLE_TAGS, recursive=False)
    if title_tag:
        return normalize_text(title_tag.get_text(" ", strip=True))
    return None

def extract_section_content(sec, tables_map, heading_level=2):
    """
    Recursively extract content from a section and all nested subsections.
    heading_level controls Markdown heading depth.
    """
    md = []

    # section title
    title = get_section_title(sec)
    if title:
        md.append("#" * heading_level + f" {title}")
        md.append("")

    # extract direct paragraphs (not from nested subsections)
    for p in sec.find_all(PARA_TAGS, recursive=False):
        txt = normalize_text(p.get_text(" ", strip=True))
        if txt:
            md.append(txt)
            md.append("")

    # extract direct tables (not nested in subsections)
    for table in sec.find_all(["ce:table", "table-wrap", "table"], recursive=False):
        cap = extract_caption(table)
        rows = extract_table_rows(table)
        if not rows:
            continue

        key = (cap or "") + "|" + "|".join(rows[0])
        if key in tables_map:
            tinfo = tables_map[key]
            idx = tinfo["table_index"]

            md.append(f"**Table {idx}.** {tinfo.get('caption','')}".strip())
            md.append("")
            md.append(rows_to_markdown(tinfo["rows"]))
            md.append("")

    # recursively extract subsections
    for child in sec.find_all(SECTION_TAGS, recursive=False):
        md.append(extract_section_content(child, tables_map, heading_level=heading_level+1))
        md.append("")

    return "\n".join(md).strip()

# -------------------------------------------------
# Main save function
# -------------------------------------------------
def save_paper_as_markdown_and_tables(doi, outdir="output"):
    paper_id = safe_id(doi)
    base_dir = os.path.join(outdir, paper_id)

    sections_dir = os.path.join(base_dir, "sections")
    table_dir = os.path.join(base_dir, "tables")

    os.makedirs(sections_dir, exist_ok=True)
    os.makedirs(table_dir, exist_ok=True)

    xml_text = fetch_xml_by_doi(doi)
    if not xml_text:
        print("❌ Failed to fetch XML full-text.")
        return

    # Save raw XML
    xml_path = os.path.join(base_dir, "paper.xml")
    with open(xml_path, "w", encoding="utf-8") as f:
        f.write(xml_text)

    soup = BeautifulSoup(xml_text, "xml")

    # Extract tables first
    tables = extract_tables_from_xml(soup)

    # Map tables for embedding
    tables_map = {}
    for t in tables:
        cap = t.get("caption") or ""
        key = cap + "|" + "|".join(t["rows"][0])
        tables_map[key] = t

    # Save each table separately
    for t in tables:
        idx = t["table_index"]
        json_path = os.path.join(table_dir, f"table_{idx:03d}.json")
        md_path = os.path.join(table_dir, f"table_{idx:03d}.md")

        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(t, f, ensure_ascii=False, indent=2)

        md_content = ""
        if t.get("caption"):
            md_content += f"**Caption:** {t['caption']}\n\n"
        md_content += rows_to_markdown(t["rows"])

        with open(md_path, "w", encoding="utf-8") as f:
            f.write(md_content)

    combined_md = []

    # ----------------------------------------
    # Abstract → stored as Section 000
    # ----------------------------------------
    abstract_text = extract_abstract_from_xml(soup)
    if abstract_text:
        abs_md = "# Abstract\n\n" + abstract_text.strip()
        abs_path = os.path.join(sections_dir, "000_Abstract.md")
        with open(abs_path, "w", encoding="utf-8") as f:
            f.write(abs_md)
        combined_md.append(abs_md)
        combined_md.append("")

    # ----------------------------------------
    # Extract top-level sections (with subsections inside)
    # ----------------------------------------
    top_sections = soup.find_all(SECTION_TAGS)
    # top_sections includes nested sections too → filter only root ones
    top_sections = [s for s in top_sections if not s.find_parent(SECTION_TAGS)]

    sec_counter = 1
    for sec in top_sections:
        title = get_section_title(sec) or f"Section_{sec_counter}"
        md_text = extract_section_content(sec, tables_map, heading_level=1)

        if not md_text.strip():
            continue

        fname = f"{sec_counter:03d}_{safe_filename(title)}.md"
        sec_path = os.path.join(sections_dir, fname)

        with open(sec_path, "w", encoding="utf-8") as f:
            f.write(md_text)

        combined_md.append(md_text)
        combined_md.append("")
        sec_counter += 1

    # Save combined file
    paper_md_path = os.path.join(base_dir, "paper.md")
    with open(paper_md_path, "w", encoding="utf-8") as f:
        f.write("\n".join(combined_md).strip())

    print(f"✅ Saved:\n- paper.xml\n- paper.md\n- {sec_counter-1} section files in sections/\n- {len(tables)} tables in tables/\n➡️ Output: {base_dir}")

# Example usage
save_paper_as_markdown_and_tables("10.1016/j.jmrt.2025.11.003")


✅ Saved:
- paper.xml
- paper.md
- 6 section files in sections/
- 3 tables in tables/
➡️ Output: output/10.1016_j.jmrt.2025.11.003


In [2]:
import os
import json
import time
import csv
import requests
from pathlib import Path

API_KEY = "2ccec2cb21139c886e65ee56ac738fa5"
BASE = Path("scopus_export")
BASE.mkdir(exist_ok=True)

SCOPUS_URL = "https://api.elsevier.com/content/search/scopus"

HEADERS = {
    "X-ELS-APIKey": API_KEY,
    "Accept": "application/json",
    "User-Agent": "Python Scopus Client",
}

QUERY = 'TITLE-ABS-KEY("crystal plasticity simulation")'
COUNT = 25
START = 0
MAX_PAGES = 1  # adjust; 20 pages * 25 = 500 results

all_entries = []
raw_pages = []

for page in range(MAX_PAGES):
    params = {
        "query": QUERY,
        "count": COUNT,
        "start": START,
        "view": "STANDARD",
    }

    r = requests.get(SCOPUS_URL, headers=HEADERS, params=params, timeout=60)
    print("HTTP:", r.status_code, "start=", START)
    r.raise_for_status()

    data = r.json()
    raw_pages.append(data)

    if "search-results" not in data:
        raise RuntimeError("Scopus API error: missing search-results")

    entries = data["search-results"].get("entry", [])
    if not entries:
        break

    all_entries.extend(entries)
    START += COUNT

    # polite pacing (helps avoid 429)
    time.sleep(0.2)

# Save raw JSON (provenance)
raw_path = BASE / "scopus_raw_pages.json"
raw_path.write_text(json.dumps(raw_pages, indent=2), encoding="utf-8")

# Extract fields and save CSV
csv_path = BASE / "scopus_results.csv"
fieldnames = [
    "title", "doi", "eid",
    "source_title", "source_id",
    "cover_date", "issn", "eissn",
    "volume", "issue", "page_range",
    "link_scopus",
]

def get_scopus_link(item):
    # links are in item.get("link", []) with @ref == "scopus"
    for lk in item.get("link", []) or []:
        if lk.get("@ref") == "scopus":
            return lk.get("@href")
    return ""

with csv_path.open("w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=fieldnames)
    w.writeheader()

    for it in all_entries:
        w.writerow({
            "title": it.get("dc:title", ""),
            "doi": it.get("prism:doi", ""),
            "eid": it.get("eid", ""),
            "source_title": it.get("prism:publicationName", ""),
            "source_id": it.get("source-id", ""),
            "cover_date": it.get("prism:coverDate", ""),
            "issn": it.get("prism:issn", ""),
            "eissn": it.get("prism:eIssn", ""),
            "volume": it.get("prism:volume", ""),
            "issue": it.get("prism:issueIdentifier", ""),
            "page_range": it.get("prism:pageRange", ""),
            "link_scopus": get_scopus_link(it),
        })

print(f"\nSaved {len(all_entries)} records")
print("Raw pages:", raw_path)
print("CSV:", csv_path)


HTTP: 200 start= 0

Saved 25 records
Raw pages: scopus_export/scopus_raw_pages.json
CSV: scopus_export/scopus_results.csv


In [3]:
# search_openalex.py
import requests

def search_openalex(query, per_page=25):
    url = "https://api.openalex.org/works"
    params = {
        "search": query,
        "per_page": per_page
    }

    r = requests.get(url, params=params)
    r.raise_for_status()
    data = r.json()

    results = []
    for w in data["results"]:
        results.append({
            "title": w.get("title"),
            "doi": w.get("doi"),  # already normalized
            "year": w.get("publication_year"),
            "journal": w.get("host_venue", {}).get("display_name")
        })

    return results


In [ ]:
# elsevier/fulltext_parser.py

import os
import re
import json
import requests
from urllib.parse import quote
from bs4 import BeautifulSoup

# --------------------------------------------------
# Helpers
# --------------------------------------------------

def safe_id(s: str) -> str:
    """Convert DOI into a filesystem-safe string."""
    return re.sub(r"[^a-zA-Z0-9._-]+", "_", s)


def safe_filename(title: str) -> str:
    """Convert section title into a filesystem-safe filename."""
    if not title:
        return "Untitled"
    title = re.sub(r"\s+", " ", title).strip()
    title = re.sub(r"[^\w\-_ .]", "_", title)
    return title[:80].strip()


def normalize_text(s: str) -> str:
    """Normalize whitespace."""
    s = re.sub(r"\s+", " ", s)
    return s.strip()


def rows_to_markdown(rows):
    """Convert table rows into a Markdown table."""
    if not rows:
        return ""

    ncol = max(len(r) for r in rows)
    norm = [r + [""] * (ncol - len(r)) for r in rows]

    header = norm[0]
    body = norm[1:] if len(norm) > 1 else []

    md = []
    md.append("| " + " | ".join(header) + " |")
    md.append("| " + " | ".join(["---"] * ncol) + " |")

    for r in body:
        md.append("| " + " | ".join(r) + " |")

    return "\n".join(md)


# --------------------------------------------------
# Elsevier API
# --------------------------------------------------

BASE_URL = "https://api.elsevier.com/content/article/doi/"


def fetch_xml_by_doi(doi, api_key, inst_token=None):
    doi_safe = quote(doi, safe="")
    url = f"{BASE_URL}{doi_safe}"

    headers = {
        "X-ELS-APIKey": api_key,
        "Accept": "text/xml",
    }
    if inst_token:
        headers["X-ELS-Insttoken"] = inst_token

    r = requests.get(url, headers=headers, params={"view": "FULL"}, timeout=30)
    if r.ok and r.text.strip():
        return r.text
    return None


# --------------------------------------------------
# Table extraction
# --------------------------------------------------

def extract_table_rows(table_tag):
    rows = []
    row_tags = table_tag.find_all(["ce:row", "row", "tr"])
    for row in row_tags:
        cells = []
        cell_tags = row.find_all(["ce:entry", "entry", "td", "th"])
        for c in cell_tags:
            cells.append(normalize_text(c.get_text(" ", strip=True)))
        if any(cells):
            rows.append(cells)
    return rows


def extract_caption(table_tag):
    cap = table_tag.find(["ce:caption", "caption", "title"])
    if cap:
        return normalize_text(cap.get_text(" ", strip=True))
    return None


def extract_tables_from_xml(soup):
    tables = soup.find_all(["ce:table", "table-wrap", "table"])
    extracted = []
    idx = 1

    for t in tables:
        rows = extract_table_rows(t)
        if not rows:
            continue

        extracted.append({
            "table_index": idx,
            "caption": extract_caption(t),
            "rows": rows,
        })
        idx += 1

    return extracted


# --------------------------------------------------
# Abstract extraction
# --------------------------------------------------

def extract_abstract_from_xml(soup):
    abs_tag = soup.find(["ce:abstract", "abstract"])
    if abs_tag:
        paras = abs_tag.find_all(["ce:para", "p"])
        if paras:
            return "\n\n".join(
                normalize_text(p.get_text(" ", strip=True))
                for p in paras if p.get_text(strip=True)
            )
        return normalize_text(abs_tag.get_text(" ", strip=True))

    dc_desc = soup.find("dc:description")
    if dc_desc and dc_desc.get_text(strip=True):
        return normalize_text(dc_desc.get_text(" ", strip=True))

    return None


# --------------------------------------------------
# Section extraction
# --------------------------------------------------

def section_to_markdown(sec, tables_map):
    md_lines = []

    # Section title
    st = sec.find(["ce:section-title", "title"])
    if st:
        md_lines.append("## " + normalize_text(st.get_text(" ", strip=True)))
        md_lines.append("")

    # Paragraphs
    for p in sec.find_all(["ce:para", "p"], recursive=True):
        txt = normalize_text(p.get_text(" ", strip=True))
        if txt:
            md_lines.append(txt)
            md_lines.append("")

    # Tables inside section
    for table in sec.find_all(["ce:table", "table-wrap", "table"], recursive=True):
        rows = extract_table_rows(table)
        if not rows:
            continue

        cap = extract_caption(table) or ""
        key = cap + "|" + "|".join(rows[0])

        if key in tables_map:
            tinfo = tables_map[key]
            idx = tinfo["table_index"]

            md_lines.append(f"### Table {idx}")
            if tinfo.get("caption"):
                md_lines.append(f"**Caption:** {tinfo['caption']}")
            md_lines.append("")
            md_lines.append(rows_to_markdown(tinfo["rows"]))
            md_lines.append("")

    return "\n".join(md_lines).strip()

# --------------------------------------------------
# Reference extraction
# --------------------------------------------------

def extract_references_from_xml(soup):
    """
    Extract reference list with numbering and DOIs from Elsevier XML.
    Returns a list of dicts with keys:
    label (e.g. "60"), ref_id, title, doi
    """
    references = []

    for ref in soup.find_all(["ce:bib-reference", "bib-reference"]):
        ref_id = ref.get("id") or ref.get("refid")

        # Extract numeric label (e.g. [60])
        label = None
        if ref_id:
            m = re.search(r"(\d+)", ref_id)
            if m:
                label = m.group(1)

        # Title
        title_tag = ref.find(["ce:title", "title"])
        title = (
            normalize_text(title_tag.get_text(" ", strip=True))
            if title_tag else None
        )

        # DOI (preferred, deterministic)
        doi_tag = ref.find("ce:doi")
        doi = doi_tag.get_text(strip=True) if doi_tag else None

        references.append({
            "label": label,     # "60"
            "ref_id": ref_id,   # "bib60"
            "title": title,
            "doi": doi
        })

    return references


# --------------------------------------------------
# Main entry point
# --------------------------------------------------

def save_paper_as_markdown_and_tables(
    doi,
    api_key,
    inst_token=None,
    outdir="output"
):
    paper_id = safe_id(doi)
    base_dir = os.path.join(outdir, paper_id)

    # --------------------------------------------------
    # Fetch XML FIRST (no folders yet)
    # --------------------------------------------------
    xml_text = fetch_xml_by_doi(doi, api_key, inst_token)
    if not xml_text:
        print(f"❌ Failed to fetch XML for DOI: {doi}")
        return

    # --------------------------------------------------
    # Create folders ONLY if fetch succeeded
    # --------------------------------------------------
    sections_dir = os.path.join(base_dir, "sections")
    tables_dir = os.path.join(base_dir, "tables")

    os.makedirs(sections_dir, exist_ok=True)
    os.makedirs(tables_dir, exist_ok=True)

    # Save raw XML
    xml_path = os.path.join(base_dir, "paper.xml")
    with open(xml_path, "w", encoding="utf-8") as f:
        f.write(xml_text)

    soup = BeautifulSoup(xml_text, "xml")

        # --------------------------------------------------
    # Extract and save references
    # --------------------------------------------------
    references = extract_references_from_xml(soup)

    if references:
        with open(
            os.path.join(base_dir, "references.json"),
            "w", encoding="utf-8"
        ) as f:
            json.dump(references, f, ensure_ascii=False, indent=2)


    # Paper title
    title_tag = soup.find(
        ["dc:title", "ce:title", "article-title", "title"]
    )
    paper_title = (
        normalize_text(title_tag.get_text(" ", strip=True))
        if title_tag else "Untitled Paper"
    )

    # Extract tables
    tables = extract_tables_from_xml(soup)

    tables_map = {}
    for t in tables:
        cap = t.get("caption") or ""
        key = cap + "|" + "|".join(t["rows"][0])
        tables_map[key] = t

    # Save tables
    for t in tables:
        idx = t["table_index"]

        with open(
            os.path.join(tables_dir, f"table_{idx:03d}.json"),
            "w", encoding="utf-8"
        ) as f:
            json.dump(t, f, ensure_ascii=False, indent=2)

        md = ""
        if t.get("caption"):
            md += f"**Caption:** {t['caption']}\n\n"
        md += rows_to_markdown(t["rows"])

        with open(
            os.path.join(tables_dir, f"table_{idx:03d}.md"),
            "w", encoding="utf-8"
        ) as f:
            f.write(md)

    # Abstract
    combined_md = [f"# {paper_title}", ""]
    abstract = extract_abstract_from_xml(soup)

    if abstract:
        abs_md = "# Abstract\n\n" + abstract
        with open(
            os.path.join(sections_dir, "000_Abstract.md"),
            "w", encoding="utf-8"
        ) as f:
            f.write(abs_md)

        combined_md.extend([abs_md, ""])

    # Sections
    sections = soup.find_all(["ce:section", "sec"])
    sec_idx = 1

    for sec in sections:
        sec_title_tag = sec.find(["ce:section-title", "title"])
        sec_title = (
            normalize_text(sec_title_tag.get_text(" ", strip=True))
            if sec_title_tag else f"Section_{sec_idx}"
        )

        md_text = section_to_markdown(sec, tables_map)
        if not md_text:
            continue

        fname = f"{sec_idx:03d}_{safe_filename(sec_title)}.md"
        with open(
            os.path.join(sections_dir, fname),
            "w", encoding="utf-8"
        ) as f:
            f.write(md_text)

        combined_md.extend([md_text, ""])
        sec_idx += 1

    # Combined Markdown
    with open(
        os.path.join(base_dir, "paper.md"),
        "w", encoding="utf-8"
    ) as f:
        f.write("\n".join(combined_md).strip())

    print(f"✅ DOI processed: {doi}")
    print(f"📂 Output directory: {base_dir}")

save_paper_as_markdown_and_tables(
    doi="10.1016/j.jmrt.2025.11.003",
    api_key="a52e01cb767cf5aa497eeef089d3595a"
)

✅ DOI processed: 10.1016/j.jmrt.2025.11.003
📂 Output directory: output/10.1016_j.jmrt.2025.11.003


In [ ]:
import re
import requests

CROSSREF_URL = "https://api.crossref.org/works"


def lookup_doi_crossref(journal=None, year=None, volume=None, article_number=None):
    """
    Deterministic DOI lookup using Crossref.
    Returns DOI string or None.
    """
    params = {
        "rows": 5,
        "select": "DOI,container-title,issued,volume,article-number"
    }

    filters = []
    if year:
        filters.append(f"from-pub-date:{year}")
        filters.append(f"until-pub-date:{year}")
    if filters:
        params["filter"] = ",".join(filters)

    if journal:
        params["query.container-title"] = journal
    if volume:
        params["query.volume"] = volume
    if article_number:
        params["query"] = article_number

    try:
        r = requests.get(CROSSREF_URL, params=params, timeout=20)
        r.raise_for_status()
        items = r.json()["message"]["items"]

        # Heuristic: exactly one strong candidate
        if len(items) == 1:
            return items[0].get("DOI")
    except Exception:
        pass

    return None


def extract_references_from_xml(soup):
    """
    Extract reference list from Elsevier XML.
    - Preserves paper-local reference numbering
    - Uses publisher DOI if present
    - Resolves missing DOIs via Crossref (deterministic)
    - Never guesses

    Returns list of dicts with keys:
      label, title, doi, doi_source, resolved
    """
    references = []

    for ref in soup.find_all(["ce:bib-reference", "bib-reference"]):
        ref_id = ref.get("id") or ref.get("refid")

        # ----------------------------
        # Reference label (e.g. [60])
        # ----------------------------
        label = None
        if ref_id:
            m = re.search(r"(\d+)", ref_id)
            if m:
                label = m.group(1)

        # ----------------------------
        # Title
        # ----------------------------
        title_tag = ref.find(["ce:title", "title"])
        title = (
            normalize_text(title_tag.get_text(" ", strip=True))
            if title_tag else None
        )

        # ----------------------------
        # DOI (publisher-provided)
        # ----------------------------
        doi_tag = ref.find("ce:doi")
        doi = doi_tag.get_text(strip=True) if doi_tag else None
        doi_source = "publisher" if doi else None
        resolved = bool(doi)

        # ----------------------------
        # If DOI missing → Crossref lookup
        # ----------------------------
        if not doi:
            journal = None
            year = None
            volume = None
            article_number = None

            # Journal
            journal_tag = ref.find("sb:maintitle")
            if journal_tag:
                journal = normalize_text(journal_tag.get_text(" ", strip=True))

            # Year
            year_tag = ref.find("sb:date")
            if year_tag:
                try:
                    year = int(year_tag.get_text(strip=True))
                except ValueError:
                    pass

            # Volume
            volume_tag = ref.find("sb:volume-nr")
            if volume_tag:
                volume = volume_tag.get_text(strip=True)

            # Article number / page
            article_tag = ref.find("sb:article-number")
            if article_tag:
                article_number = article_tag.get_text(strip=True)

            doi = lookup_doi_crossref(
                journal=journal,
                year=year,
                volume=volume,
                article_number=article_number
            )

            if doi:
                doi_source = "crossref"
                resolved = True
            else:
                doi_source = "unresolved"
                resolved = False

        references.append({
            "label": label,            # e.g. "60"
            "title": title,
            "doi": doi,                # may be None
            "doi_source": doi_source,  # publisher / crossref / unresolved
            "resolved": resolved
        })

    return references


In [14]:
import requests

url = "https://api.crossref.org/works"
params = {
    "query.bibliographic": "Acta Mater. 214 (2021) 116986",
    "rows": 1
}

r = requests.get(url, params=params)
doi = r.json()["message"]["items"][0]["DOI"]
print(doi)


10.1016/j.actamat.2021.116986


In [ ]:
# elsevier/fulltext_parser.py

import os
import re
import json
import requests
from urllib.parse import quote
from bs4 import BeautifulSoup

# ==================================================
# Helpers
# ==================================================

def safe_id(s: str) -> str:
    return re.sub(r"[^a-zA-Z0-9._-]+", "_", s)


def safe_filename(title: str) -> str:
    if not title:
        return "Untitled"
    title = re.sub(r"\s+", " ", title).strip()
    title = re.sub(r"[^\w\-_ .]", "_", title)
    return title[:80].strip()


def normalize_text(s: str) -> str:
    return re.sub(r"\s+", " ", s).strip()


def rows_to_markdown(rows):
    if not rows:
        return ""
    ncol = max(len(r) for r in rows)
    rows = [r + [""] * (ncol - len(r)) for r in rows]
    md = [
        "| " + " | ".join(rows[0]) + " |",
        "| " + " | ".join(["---"] * ncol) + " |",
    ]
    for r in rows[1:]:
        md.append("| " + " | ".join(r) + " |")
    return "\n".join(md)

# ==================================================
# Elsevier API
# ==================================================

BASE_URL = "https://api.elsevier.com/content/article/doi/"

def fetch_xml_by_doi(doi, api_key, inst_token=None):
    url = f"{BASE_URL}{quote(doi, safe='')}"
    headers = {
        "X-ELS-APIKey": api_key,
        "Accept": "text/xml",
    }
    if inst_token:
        headers["X-ELS-Insttoken"] = inst_token

    r = requests.get(url, headers=headers, params={"view": "FULL"}, timeout=30)
    if r.ok and r.text.strip():
        return r.text
    return None

# ==================================================
# Table extraction
# ==================================================

def extract_table_rows(table_tag):
    rows = []
    for row in table_tag.find_all(["ce:row", "row", "tr"]):
        cells = [
            normalize_text(c.get_text(" ", strip=True))
            for c in row.find_all(["ce:entry", "entry", "td", "th"])
        ]
        if any(cells):
            rows.append(cells)
    return rows


def extract_caption(table_tag):
    cap = table_tag.find(["ce:caption", "caption", "title"])
    return normalize_text(cap.get_text(" ", strip=True)) if cap else None


def extract_tables_from_xml(soup):
    tables, idx = [], 1
    for t in soup.find_all(["ce:table", "table-wrap", "table"]):
        rows = extract_table_rows(t)
        if not rows:
            continue
        tables.append({
            "table_index": idx,
            "caption": extract_caption(t),
            "rows": rows,
        })
        idx += 1
    return tables

# ==================================================
# Citation normalization (GLOBAL)
# ==================================================

def build_bib_id_to_index_map(soup):
    ref_map = {}
    for ref in soup.find_all(["ce:bib-reference", "bib-reference"]):
        refid = ref.get("id")
        if not refid:
            continue

        label_tag = ref.find("ce:label")
        if label_tag:
            m = re.search(r"\d+", label_tag.get_text())
            if m:
                ref_map[refid] = m.group(0)
                continue

        m = re.search(r"(\d+)", refid)
        if m:
            ref_map[refid] = str(int(m.group(1)))

    return ref_map


def normalize_all_cross_refs(soup):
    """
    Replace ALL <ce:cross-ref> in the document with numeric [n].
    Applies to abstract, body, tables, captions, etc.
    """
    ref_map = build_bib_id_to_index_map(soup)

    for cr in soup.find_all("ce:cross-ref"):
        refid = cr.get("refid")
        if refid and refid in ref_map:
            cr.replace_with(f"[{ref_map[refid]}]")
        else:
            cr.replace_with("[?]")

    return ref_map

# ==================================================
# Abstract & section extraction
# ==================================================

def extract_abstract_from_xml(soup):
    abs_tag = soup.find(["ce:abstract", "abstract"])
    if not abs_tag:
        return None

    paras = abs_tag.find_all(["ce:para", "p"])
    texts = [
        normalize_text(p.get_text(" ", strip=True))
        for p in paras if p.get_text(strip=True)
    ]
    return "\n\n".join(texts) if texts else None


def section_to_markdown(sec, tables_map):
    md = []

    title_tag = sec.find(["ce:section-title", "title"])
    if title_tag:
        md.append("## " + normalize_text(title_tag.get_text(" ", strip=True)))
        md.append("")

    for p in sec.find_all(["ce:para", "p"], recursive=True):
        txt = normalize_text(p.get_text(" ", strip=True))
        if txt:
            md.append(txt)
            md.append("")

    for table in sec.find_all(["ce:table", "table-wrap", "table"], recursive=True):
        rows = extract_table_rows(table)
        if not rows:
            continue
        cap = extract_caption(table) or ""
        key = cap + "|" + "|".join(rows[0])
        if key in tables_map:
            t = tables_map[key]
            md.append(f"### Table {t['table_index']}")
            if t.get("caption"):
                md.append(f"**Caption:** {t['caption']}")
            md.append("")
            md.append(rows_to_markdown(t["rows"]))
            md.append("")

    return "\n".join(md).strip()

# ==================================================
# Reference extraction + Crossref DOI resolution
# ==================================================

CROSSREF_URL = "https://api.crossref.org/works"

def lookup_doi_crossref_biblio(journal, volume, year, article_number):
    query = f"{journal} {volume} ({year}) {article_number}"
    try:
        r = requests.get(
            CROSSREF_URL,
            params={"query.bibliographic": query, "rows": 1},
            timeout=20,
        )
        r.raise_for_status()
        items = r.json().get("message", {}).get("items", [])
        return items[0].get("DOI") if items else None
    except Exception:
        return None


def extract_references_from_xml(soup):
    refs = []
    for ref in soup.find_all(["ce:bib-reference", "bib-reference"]):
        refid = ref.get("id")
        label = re.search(r"\d+", refid).group(0) if refid else None

        title_tag = ref.find(["ce:title", "title"])
        title = normalize_text(title_tag.get_text(" ", strip=True)) if title_tag else None

        doi_tag = ref.find("ce:doi")
        doi = doi_tag.get_text(strip=True) if doi_tag else None

        if not doi:
            journal = ref.find_text("sb:maintitle")
            volume = ref.find_text("sb:volume-nr")
            year = ref.find_text("sb:date")
            article = ref.find_text("sb:article-number")
            if journal and volume and year and article:
                doi = lookup_doi_crossref_biblio(journal, volume, year, article)

        refs.append({"label": label, "title": title, "doi": doi})

    return refs

# ==================================================
# Main entry point
# ==================================================

def save_paper_as_markdown_and_tables(
    doi,
    api_key,
    inst_token=None,
    outdir="data/fulltext"
):
    xml = fetch_xml_by_doi(doi, api_key, inst_token)
    if not xml:
        print(f"❌ Failed to fetch XML for {doi}")
        return

    base = os.path.join(outdir, safe_id(doi))
    os.makedirs(base, exist_ok=True)

    with open(os.path.join(base, "paper.xml"), "w", encoding="utf-8") as f:
        f.write(xml)

    soup = BeautifulSoup(xml, "xml")

    # 🔥 GLOBAL citation normalization (ONE TIME)
    normalize_all_cross_refs(soup)

    refs = extract_references_from_xml(soup)
    with open(os.path.join(base, "references.json"), "w", encoding="utf-8") as f:
        json.dump(refs, f, indent=2)

    tables = extract_tables_from_xml(soup)
    tables_map = {
        (t.get("caption") or "") + "|" + "|".join(t["rows"][0]): t
        for t in tables
    }

    os.makedirs(os.path.join(base, "tables"), exist_ok=True)
    for t in tables:
        idx = t["table_index"]
        with open(os.path.join(base, "tables", f"table_{idx:03d}.md"), "w") as f:
            f.write(rows_to_markdown(t["rows"]))

    sections_dir = os.path.join(base, "sections")
    os.makedirs(sections_dir, exist_ok=True)

    combined = []
    title_tag = soup.find(["dc:title", "ce:title", "article-title", "title"])
    title = normalize_text(title_tag.get_text()) if title_tag else "Untitled Paper"
    combined.append(f"# {title}\n")

    abstract = extract_abstract_from_xml(soup)
    if abstract:
        with open(os.path.join(sections_dir, "000_Abstract.md"), "w") as f:
            f.write("# Abstract\n\n" + abstract)
        combined.append("# Abstract\n\n" + abstract + "\n")

    i = 1
    for sec in soup.find_all(["ce:section", "sec"]):
        md = section_to_markdown(sec, tables_map)
        if md:
            with open(os.path.join(sections_dir, f"{i:03d}.md"), "w") as f:
                f.write(md)
            combined.append(md + "\n")
            i += 1

    with open(os.path.join(base, "paper.md"), "w") as f:
        f.write("\n".join(combined))

    print(f"✅ Parsed: {doi}")
    print(f"📂 Output: {base}")


<ce:bib-reference id="bib1">
<ce:label>Agaram et al., 2021</ce:label>
<sb:reference id="sref1">
<sb:contribution langtype="en">
<sb:authors>
<sb:author>
<ce:given-name>S.</ce:given-name>
<ce:surname>Agaram</ce:surname>
</sb:author>
<sb:author>
<ce:given-name>A.K.</ce:given-name>
<ce:surname>Kanjarla</ce:surname>
</sb:author>
<sb:author>
<ce:given-name>B.</ce:given-name>
<ce:surname>Bhuvaraghan</ce:surname>
</sb:author>
<sb:author>
<ce:given-name>S.M.</ce:given-name>
<ce:surname>Srinivasan</ce:surname>
</sb:author>
</sb:authors>
<sb:title>
<sb:maintitle>Dislocation density based crystal plasticity model incorporating the effect of precipitates in IN718 under monotonic and cyclic deformation</sb:maintitle>
</sb:title>
</sb:contribution>
<sb:host>
<sb:issue>
<sb:series>
<sb:title>
<sb:maintitle>Int. J. Plast.</sb:maintitle>
</sb:title>
<sb:volume-nr>141</sb:volume-nr>
</sb:series>
<sb:date>2021</sb:date>
</sb:issue>
<sb:article-number>102990</sb:article-number>
</sb:host>
</sb:reference>


TypeError: 'NoneType' object is not callable